In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures



In [ ]:
#path = r"C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\"
path = r"C:\Users\samcl\Massachusetts Institute of Technology\Truck Parking Capstone - Truck Stop Finder 🚚⛽\\"

In [ ]:
# Sourced directly from TruckerPath
park_data_1 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_1 - Copy.csv")
park_data_2 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_2 - Copy.csv")
park_data_3 = pd.read_csv(
    path + r"5. Source & Refrence Files\0. TruckerPath Data\MIT_2025_High_Volume_Routes_Parking_Data_3 - Copy.csv")
park_data = pd.concat([park_data_1, park_data_2, park_data_3], ignore_index=True)

In [ ]:
park_data

In [ ]:
truck_stop = pd.read_excel("output_excel\Model_Stops_V3.xlsx")

In [ ]:
truck_stop


In [ ]:
avail_park = park_data[park_data["pin id"].isin(truck_stop["pin id"].unique())].copy()

In [ ]:
avail_park

In [ ]:
avail_park["time(utc)"] = pd.to_datetime(avail_park["time(utc)"])

In [ ]:
avail_park["day_of_week"] = avail_park["time(utc)"].dt.dayofweek
avail_park["day_name"] = avail_park["time(utc)"].dt.day_name()
avail_park["hour"] = avail_park["time(utc)"].dt.hour
avail_park["month"] = avail_park["time(utc)"].dt.month

In [ ]:
avail_park

In [ ]:
avail_park.groupby(['pin id', 'hour']).agg({'parking status': 'count'}).reset_index()

In [ ]:
avail_park['parking status'].unique()


In [ ]:
avail_park.groupby('parking status').agg({'city': 'count'}).reset_index()

In [ ]:
avail_park = avail_park[avail_park["parking status"] != 'Paid'].copy()

In [ ]:
avail_park['percent_avail'] = 0

In [ ]:
avail_park['percent_avail'] = np.where(avail_park['parking status'] == 'Full', .1, avail_park['percent_avail'])
avail_park['percent_avail'] = np.where(avail_park['parking status'] == 'Some', .5, avail_park['percent_avail'])
avail_park['percent_avail'] = np.where(avail_park['parking status'] == 'Lots', .9, avail_park['percent_avail'])


In [ ]:
avail_park

In [ ]:
reg_df = avail_park[avail_park['pin id'] == 'cf1f78fe923afe05f7597da2be7a3da8']

In [ ]:
reg_df

In [ ]:
reg_df.to_excel('TA_Madison_45.xlsx')

In [ ]:
train_set = reg_df.iloc[0:int(reg_df.shape[0] * .8)]
test_set = reg_df.iloc[int(reg_df.shape[0] * .8):]

print("Train set shape:", train_set.shape)
print("Test set shape:", test_set.shape)

In [ ]:
y_train = train_set['percent_avail']
X_train = train_set[['hour', 'day_of_week', 'month']]
y_test = test_set['percent_avail']
X_test = test_set[['hour', 'day_of_week', 'month']]

In [ ]:
# y_train = train_set['percent_avail']
# X_train = train_set[['hour']]

X_train_cons = sm.add_constant(X_train)

avail_ols = sm.OLS(y_train, X_train_cons).fit()

summary = avail_ols.summary()
print(summary)

In [ ]:
def calculate_mape(y_true, y_pred):
    return (100 * abs((y_true - y_pred) / y_true)).mean()


In [ ]:
#polynomial with only important features

poly = PolynomialFeatures(degree=8, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
feature_names = poly.get_feature_names_out(['hour', 'day_of_week', 'month'])
X_train_poly_df = pd.DataFrame(X_train_poly, columns=feature_names, index=X_train.index)
X_test_poly_df = pd.DataFrame(X_test_poly, columns=feature_names, index=X_test.index)

X_train_poly_df = sm.add_constant(X_train_poly_df)
X_test_poly_df = sm.add_constant(X_test_poly_df)

model = sm.OLS(y_train, X_train_poly_df).fit()
y_pred_poly = model.predict(X_test_poly_df)

print(model.summary())

print("Poly  MAPE:", calculate_mape(y_test, y_pred_poly))
print("Poly RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_poly)))

In [ ]:
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("R²:", r2_score(y_test, y_pred))


In [ ]:
# Create the model
model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

# Fit the model
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R²:", r2)
print("RMSE:", rmse)


In [ ]:
final_df = pd.DataFrame()
for p in avail_park['pin id'].unique():
    reg_df = avail_park[avail_park['pin id'] == p].copy()

    train_set = reg_df.iloc[0:int(reg_df.shape[0] * .8)]
    test_set = reg_df.iloc[int(reg_df.shape[0] * .8):]
    if test_set.shape[0] > 0 and train_set.shape[0] > 0:
        y_train = train_set['percent_avail']
        X_train = train_set[['hour', 'day_of_week']]
        y_test = test_set['percent_avail']
        X_test = test_set[['hour', 'day_of_week']]

        rf = RandomForestRegressor(
            n_estimators=400,
            max_depth=None,
            random_state=42
        )
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_test)
        data = {'pin id': [p],
                'R^2': [r2_score(y_test, y_pred)],
                'Sample_count': [reg_df.shape[0]], }
        temp_df = pd.DataFrame(data=data)
        final_df = pd.concat([final_df, temp_df], ignore_index=True)

In [ ]:
final_df

In [ ]:
final_df.to_csv('Final_Regression_Avail.csv', index=False)